# 01. Temporal Patterns: When Does Yogyakarta Move?

**Data sources:** `presence_kapanewon_hourly.csv`, `tier1_summary.csv`

**Questions**
1. How does the size of the observed device sample change over the 151 days, and why must we normalise?
2. What does a typical 24-hour activity cycle look like, and how do weekdays differ from weekends?
3. Which day-of-week and hour combinations are the busiest?
4. Did the Christmas and New Year period change the daily rhythm?

In [ ]:
# @title 0. Setup (jalankan sekali, otomatis di Google Colab)
# Di Colab: membuat folder kerja, menulis modul helper, dan mengunduh data dari Zenodo.
# Di laptop (repo GitHub): sel ini tidak melakukan apa-apa.
import os, sys, hashlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    BASE = "/content/jogjapulse"
    for d in ["src", "data", "figures", "notebooks"]:
        os.makedirs(f"{BASE}/{d}", exist_ok=True)
    open(f"{BASE}/src/__init__.py", "w").close()
    with open(f"{BASE}/src/jogjapulse.py", "w") as f:
        f.write('"""\nShared helpers for the JogjaPulse human mobility project.\n\nAll notebooks import from here so that loading, cleaning and normalisation\nare done the same way everywhere.\n\nDataset: Widyawan et al. (2026). JogjaPulse: An Aggregated Spatio-Temporal\nMobility Dataset for Yogyakarta, Indonesia. Zenodo.\nhttps://doi.org/10.5281/zenodo.23157460  (CC BY 4.0)\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n# --------------------------------------------------------------------------\n# Paths\n# --------------------------------------------------------------------------\nROOT = Path(__file__).resolve().parents[1]\nDATA_DIR = ROOT / "data"\nFIG_DIR = ROOT / "figures"\nFIG_DIR.mkdir(exist_ok=True)\n\n# --------------------------------------------------------------------------\n# Constants from the dataset README\n# --------------------------------------------------------------------------\nGOOD_PERIOD_END = "2022-02-28"      # OD flows after this are reduced quality\nSAMPLE_BREAKS = ["2022-01-01", "2022-02-16"]  # abrupt drops in device sample\nANOMALY_DAYS = ["2022-03-08"]       # one-day delivery issue\nNIGHT_HOURS = list(range(0, 6))     # 00:00-05:59 WIB\nDAY_HOURS = list(range(10, 17))     # 10:00-16:59 WIB\nHOLIDAY_WINDOW = ("2021-12-24", "2022-01-02")  # Christmas and New Year\nKAB_ORDER = ["YOGYAKARTA", "SLEMAN", "BANTUL", "KULON PROGO", "GUNUNGKIDUL"]\nDOW_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]\n\n\ndef _path(name: str) -> Path:\n    p = DATA_DIR / name\n    if not p.exists():\n        raise FileNotFoundError(\n            f"{p} not found. Run `python scripts/download_data.py` first "\n            "or download the CSV files from Zenodo into the data/ folder."\n        )\n    return p\n\n\n# --------------------------------------------------------------------------\n# Loaders\n# --------------------------------------------------------------------------\ndef load_hourly(drop_anomalies: bool = True) -> pd.DataFrame:\n    """Kapanewon x hour presence (n_maids, n_pings)."""\n    df = pd.read_csv(_path("presence_kapanewon_hourly.csv"),\n                     dtype={"kab_code": str, "kapanewon_code": str})\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    df["dow"] = df["date"].dt.dayofweek          # 0 = Monday\n    df["is_weekend"] = df["dow"] >= 5\n    df["month"] = df["date"].dt.to_period("M").astype(str)\n    return df\n\n\ndef load_cells(drop_anomalies: bool = True) -> pd.DataFrame:\n    """500 m grid cell x day presence (n_maids)."""\n    df = pd.read_csv(_path("presence_cell500m_daily.csv"))\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    return df\n\n\ndef load_od() -> pd.DataFrame:\n    """Monthly home-based OD visitor counts, joined with origin denominators."""\n    od = pd.read_csv(_path("od_home_visits_kapanewon_monthly.csv"),\n                     dtype={"origin_code": str, "dest_code": str,\n                            "origin_kab_code": str, "dest_kab_code": str})\n    org = pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                      dtype={"origin_code": str, "origin_kab_code": str})\n    od = od.merge(org[["month", "origin_code", "n_homed_maids",\n                       "n_homed_maids_daytime"]],\n                  on=["month", "origin_code"], how="left")\n    od["share"] = od["n_maids"] / od["n_homed_maids_daytime"]\n    od["is_internal"] = od["origin_code"] == od["dest_code"]\n    od["reduced_quality"] = od["reduced_quality"].astype(str).str.lower() == "true"\n    return od\n\n\ndef load_origins() -> pd.DataFrame:\n    return pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                       dtype={"origin_code": str, "origin_kab_code": str})\n\n\ndef load_home_vs_population() -> pd.DataFrame:\n    return pd.read_csv(_path("home_maids_vs_population_kapanewon.csv"))\n\n\ndef load_night_validation() -> pd.DataFrame:\n    return pd.read_csv(_path("validation_night_presence_kapanewon.csv"))\n\n\n# --------------------------------------------------------------------------\n# Normalisation (README section 6: the device sample shrinks over time,\n# so raw counts must never be compared across dates)\n# --------------------------------------------------------------------------\ndef add_hourly_share(df: pd.DataFrame) -> pd.DataFrame:\n    """Share of each kapanewon in the regional total of the same hour.\n\n    Note: n_maids is a distinct count, so the regional sum is not the exact\n    number of distinct devices. It is used here only as a common denominator\n    that removes the effect of the shrinking sample.\n    """\n    out = df.copy()\n    tot = out.groupby(["date", "hour"])["n_maids"].transform("sum")\n    out["share_hour"] = out["n_maids"] / tot\n    return out\n\n\ndef add_daily_share(cells: pd.DataFrame) -> pd.DataFrame:\n    """Share of each grid cell in the total presence of the same day."""\n    out = cells.copy()\n    out["share_day"] = out["n_maids"] / out.groupby("date")["n_maids"].transform("sum")\n    return out\n\n\ndef hourly_profile(df: pd.DataFrame, value: str = "n_pings") -> pd.DataFrame:\n    """Percentage of each day\'s activity that falls in each hour.\n\n    n_pings is additive, so it is the correct variable for regional totals.\n    Returns one row per date with 24 hour columns.\n    """\n    daily = df.groupby(["date", "hour"])[value].sum().unstack("hour").fillna(0)\n    return daily.div(daily.sum(axis=1), axis=0) * 100\n\n\ndef kapanewon_profiles(df: pd.DataFrame) -> pd.DataFrame:\n    """24-hour shape per kapanewon (each row sums to 1), based on the mean\n    hourly share. Used for functional-zone clustering."""\n    s = add_hourly_share(df)\n    prof = s.groupby(["kapanewon", "hour"])["share_hour"].mean().unstack("hour")\n    prof = prof.reindex(columns=range(24))\n    prof = prof.interpolate(axis=1, limit_direction="both")\n    return prof.div(prof.sum(axis=1), axis=0)\n\n\ndef savefig(fig, name: str, dpi: int = 200) -> Path:\n    path = FIG_DIR / name\n    fig.savefig(path, dpi=dpi, bbox_inches="tight")\n    return path\n')
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "plotly>=5.24"], check=False)
    import requests
    FILES = {'presence_cell500m_daily.csv': '0b4d544e8c9ab26d69360821e9ebff27', 'presence_kapanewon_hourly.csv': '90a4a8ae6eb68abc68f7a1021a51cce8', 'od_home_visits_kapanewon_monthly.csv': '5857291c608dd5335f6d8833c1ec7b7a', 'od_origin_homes_monthly.csv': 'd14368c48d3937c9146ed9054aee3c79', 'od_summary.csv': '139962d1c8101eb6873be6f902320efb', 'home_maids_vs_population_kapanewon.csv': '0f01f7a3bb7027174cc1b5e1ad0e8224', 'validation_night_presence_kapanewon.csv': '4a8d0f393fd6d4fa17fb0e60c6f7408e', 'tier1_summary.csv': 'd391339521ce3714f1616c430512b0e0'}
    for name, md5 in FILES.items():
        target = f"{BASE}/data/{name}"
        if os.path.exists(target):
            continue
        print("Downloading", name, "...", end=" ", flush=True)
        r = requests.get("https://zenodo.org/records/23157460/files/" + name, params={"download": 1}, timeout=300)
        r.raise_for_status()
        open(target, "wb").write(r.content)
        print("OK" if hashlib.md5(r.content).hexdigest() == md5 else "checksum mismatch")
    os.chdir(f"{BASE}/notebooks")
    print("Setup selesai. Folder kerja:", BASE)


In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import jogjapulse as jp

sns.set_theme(style="whitegrid", context="talk", font_scale=0.8)
hourly = jp.load_hourly()
print(hourly.shape)
hourly.head()

## 1.1 Data quality: suppression and a shrinking sample
Rows with fewer than 5 devices were removed (k-anonymity). A missing row means "fewer than 5", not zero.

In [ ]:
tier1 = pd.read_csv(jp.DATA_DIR / "tier1_summary.csv")
tier1

In [ ]:
daily = (hourly.groupby("date")
         .agg(total_pings=("n_pings", "sum"),
              max_hourly_devices=("n_maids", "max"),
              rows=("n_maids", "size"))
         .reset_index())

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(daily["date"], daily["total_pings"] / 1e3, color="#1f4e79", lw=1.8)
for d in jp.SAMPLE_BREAKS:
    ax.axvline(pd.Timestamp(d), color="#c0392b", ls="--", lw=1.2)
    ax.text(pd.Timestamp(d), ax.get_ylim()[1] * 0.92, f" sample drop\n {d}",
            color="#c0392b", fontsize=9, va="top")
ax.set_title("Daily pings in DIY (raw): the device sample shrinks over time")
ax.set_ylabel("Pings per day (thousands)")
ax.set_xlabel("")
jp.savefig(fig, "01_daily_pings_raw.png")
plt.show()

monthly = daily.assign(month=daily["date"].dt.to_period("M")).groupby("month")["total_pings"].mean().round()
print("Mean pings per day by month:\n", monthly)

**Takeaway:** the drop is caused by changes in the data supplier's device panel, not by people staying home.
Every comparison across time below therefore uses *shares* (percent of the day's or hour's total) instead of raw counts.

## 1.2 The 24-hour cycle: weekday vs weekend

In [ ]:
prof = jp.hourly_profile(hourly)                # % of each day's pings per hour
dow = pd.Series(prof.index.dayofweek, index=prof.index)
wd = prof[dow < 5].mean()
we = prof[dow >= 5].mean()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(wd.index, wd.values, marker="o", label="Weekday", color="#1f4e79")
ax.plot(we.index, we.values, marker="s", label="Weekend", color="#e67e22")
ax.fill_between(wd.index, wd.values, we.values, color="grey", alpha=0.15)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day (WIB)")
ax.set_ylabel("% of daily pings")
ax.set_title("Typical daily activity cycle in DIY")
ax.legend()
jp.savefig(fig, "01_diurnal_weekday_weekend.png")
plt.show()

summary = pd.DataFrame({"weekday_%": wd.round(2), "weekend_%": we.round(2)})
print("Quietest hour:", wd.idxmin(), "| Busiest hour:", wd.idxmax())
summary.T

## 1.3 Heatmap: day of week x hour

In [ ]:
prof_long = prof.stack().rename("pct").reset_index()
prof_long["dow"] = prof_long["date"].dt.dayofweek
heat = prof_long.pivot_table(index="dow", columns="hour", values="pct", aggfunc="mean")
heat.index = [jp.DOW_NAMES[i] for i in heat.index]

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(heat, cmap="YlOrRd", ax=ax, cbar_kws={"label": "% of daily pings"})
ax.set_title("When is DIY most active? (share of daily pings)")
ax.set_xlabel("Hour of day (WIB)")
ax.set_ylabel("")
jp.savefig(fig, "01_heatmap_dow_hour.png")
plt.show()

## 1.4 Holiday effect: Christmas and New Year vs ordinary December days

In [ ]:
start, end = map(pd.Timestamp, jp.HOLIDAY_WINDOW)
is_hol = (prof.index >= start) & (prof.index <= end)
is_base = (prof.index >= "2021-12-01") & (prof.index <= "2021-12-20") & (prof.index.dayofweek < 5)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(range(24), prof[is_base].mean(), marker="o", label="Ordinary Dec weekdays (1-20 Dec)")
ax.plot(range(24), prof[is_hol].mean(), marker="s", label="Holiday window (24 Dec - 2 Jan)")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day (WIB)")
ax.set_ylabel("% of daily pings")
ax.set_title("Holiday vs ordinary days: shift in the daily rhythm")
ax.legend()
jp.savefig(fig, "01_holiday_vs_ordinary.png")
plt.show()

# New Year's Eve night: share of pings between 22:00 and 01:59
nye = prof.loc[prof.index == "2021-12-31", [22, 23]].sum(axis=1)
print("Share of pings at 22-23h on 31 Dec:", nye.round(2).tolist())
print("Average share at 22-23h on ordinary weekdays:", prof.loc[is_base, [22, 23]].sum(axis=1).mean().round(2))

## Key findings (fill in after running)
- The sample shrinks from about ... pings/day in Dec 2021 to about ... in Apr 2022, so raw counts are misleading.
- Activity is lowest around ...:00 and plateaus from 11:00 to 20:00.
- Weekends start later in the morning and have relatively more late-night activity.
- Note: the bump at 01:00 is likely a data-delivery artefact; mention it as a limitation.

In [ ]:
# @title Unduh semua grafik (Colab)
# Membuat figures.zip lalu mengunduhnya ke laptop, untuk slide dan untuk di-push ke GitHub.
if IN_COLAB:
    import shutil
    from google.colab import files as colab_files
    shutil.make_archive("/content/figures", "zip", "/content/jogjapulse/figures")
    colab_files.download("/content/figures.zip")
